**Mount Drive**

In [24]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
BASE = "/content/drive/MyDrive/cyber_fusion_models"



Mounted at /content/drive


**Imports**

In [25]:
import os, json, zipfile, math, io
from collections import Counter

import numpy as np
import pandas as pd
import joblib

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from PIL import Image
import torchvision.transforms as T


**Router Paths**

In [35]:

ROUTER_PATH       = f"{BASE}/input_type_classifier.pkl"
assert os.path.exists(ROUTER_PATH), "Missing input type classifier"
print("✓ Input type classifier loaded")

TAB_ARTIFACT_PATH = f"{BASE}/tabular_model.pkl"
TEXT_DIR          = f"{BASE}/text_branch_artifact"
IMAGE_DIR         = f"{BASE}/image_branch_artifact"
UNSEEN_ZIP        = f"{BASE}/unseen_images.zip"


✓ Input type classifier loaded


In [36]:
for p in [ROUTER_PATH, TAB_ARTIFACT_PATH, TEXT_DIR, IMAGE_DIR]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing artifact: {p}")

router = joblib.load(ROUTER_PATH)

tab_art = joblib.load(TAB_ARTIFACT_PATH)
tab_preprocess = tab_art["preprocess"]
tab_selector   = tab_art["selector"]
tab_model      = tab_art["model"]
tab_threshold  = float(tab_art.get("threshold", 0.5))


**Tabular Inference function**

In [37]:
def tabular_branch_predict(sample_path):
    df = pd.read_csv(sample_path)
    Xp = tab_preprocess.transform(df)
    Xs = tab_selector.transform(Xp)
    prob = tab_model.predict_proba(Xs)[0, 1]
    return float(prob)


**Image Branch Inference**

**Text Branch Inference**

**Structural Feature Extraction**

In [39]:
def shannon_entropy(data: bytes):
    if not data:
        return 0.0
    counts = Counter(data)
    probs = [c / len(data) for c in counts.values()]
    return -sum(p * math.log2(p) for p in probs)

def extract_structural_features(source, max_bytes=2_000_000):
    """
    source:
      - str → normal filesystem path (unseen file)
      - tuple(zip_path, inner_path) → ZIP-based input
    """

    if isinstance(source, tuple):
        zip_path, inner_path = source
        if not zipfile.is_zipfile(zip_path):
            raise ValueError(f"Not a ZIP file: {zip_path}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            if inner_path not in zf.namelist():
                raise ValueError(f"'{inner_path}' not found inside ZIP")
            raw = zf.read(inner_path)[:max_bytes]

    elif isinstance(source, str):
        if not os.path.exists(source):
            raise ValueError(f"File does not exist: {source}")
        with open(source, "rb") as f:
            raw = f.read(max_bytes)

    else:
        raise TypeError("sample must be a file path (str) or (zip_path, inner_path) tuple")

    size = len(raw)
    entropy = shannon_entropy(raw)
    ascii_ratio = sum(32 <= b <= 126 for b in raw) / max(1, size)
    byte_var = float(np.var(np.frombuffer(raw, dtype=np.uint8))) if size > 0 else 0.0

    text = raw.decode(errors="ignore")
    line_count = text.count("\n")
    comma_count = text.count(",")
    token_count = len(text.split())
    is_png = int(raw.startswith(b"\x89PNG"))

    return [size, entropy, ascii_ratio, byte_var, line_count, comma_count, token_count, is_png]


**Loading Text Branch**

In [40]:
# ===============================
# TEXT BRANCH LOADING (INFERENCE)
# ===============================

if not os.path.exists(TEXT_DIR):
    raise FileNotFoundError(f"Text branch artifact not found: {TEXT_DIR}")

meta_path = os.path.join(TEXT_DIR, "meta.json")

# Default metadata (safe fallback)
text_meta = {
    "max_len": 512,
    "positive_class": 1,
    "threshold": 0.5
}

# Load metadata if present
if os.path.exists(meta_path):
    with open(meta_path, "r") as f:
        text_meta.update(json.load(f))

# Load tokenizer and model ONCE
tokenizer_text = AutoTokenizer.from_pretrained(TEXT_DIR)
text_model = AutoModelForSequenceClassification.from_pretrained(TEXT_DIR)

# Device isolation for text branch
text_device = "cuda" if torch.cuda.is_available() else "cpu"
text_model.to(text_device)
text_model.eval()

print("Text branch loaded successfully")
print("Text device:", text_device)
print("Text metadata:", text_meta)


Text branch loaded successfully
Text device: cpu
Text metadata: {'max_len': 512, 'positive_class': 1, 'threshold': 0.5, 'model_type': 'transformer'}


In [41]:
def text_branch_predict(sample):
    """
    Inference-only text branch prediction.
    Expects a text file path.
    """

    if isinstance(sample, tuple):
        raise ValueError("Text branch does not support ZIP input")

    with open(sample, "r", errors="ignore") as f:
        text = f.read()

    inputs = tokenizer_text(
        text,
        truncation=True,
        padding=True,
        max_length=text_meta["max_len"],
        return_tensors="pt"
    )

    inputs = {k: v.to(text_device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = text_model(**inputs)
        logits = outputs.logits

        if logits.shape[-1] == 1:
            prob = torch.sigmoid(logits)[0, 0].item()
        else:
            prob = torch.softmax(logits, dim=-1)[0, text_meta["positive_class"]].item()

    return float(prob)


In [42]:
# =========================================
# IMAGE MODEL ARCHITECTURE (INFERENCE)
# =========================================

import torch
import torch.nn as nn
from torchvision import models

def build_image_model():
    """
    Reconstructs the ResNet50 malware classifier
    EXACTLY as trained in Image_Branch_Draft1.ipynb
    """

    # Load ImageNet-pretrained ResNet50
    model = models.resnet50(pretrained=True)

    # Freeze backbone (not strictly required for inference,
    # but keeps architecture identical to training)
    for param in model.parameters():
        param.requires_grad = False

    # Rebuild classifier head (MUST MATCH TRAINING)
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, 1)   # single logit (binary classification)
    )

    return model


**Load Image Branch**

In [43]:
# =========================================
# IMAGE BRANCH LOADING (INFERENCE ONLY)
# =========================================

# ---- Validate required artifacts ----
required_files = [
    os.path.join(IMAGE_DIR, "meta.json"),
    os.path.join(IMAGE_DIR, "model.pt")
]

for p in required_files:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing image artifact file: {p}")

# ---- Load metadata ----
with open(os.path.join(IMAGE_DIR, "meta.json"), "r") as f:
    image_meta = json.load(f)

# ---- Rebuild model architecture (MUST match training) ----
# IMPORTANT: build_image_model() must be defined in a previous cell
image_model = build_image_model()

try:
    image_model.load_state_dict(
        torch.load(os.path.join(IMAGE_DIR, "model.pt"), map_location="cpu")
    )
except Exception as e:
    raise RuntimeError("Failed to load image model weights. Architecture mismatch.") from e

# ---- Device placement ----
image_device = "cuda" if torch.cuda.is_available() else "cpu"
image_model.to(image_device)
image_model.eval()  # inference-only

# ---- Extract preprocessing metadata ----
img_size = int(image_meta.get("img_size", 224))
mean = image_meta.get("mean", [0.485, 0.456, 0.406])
std  = image_meta.get("std",  [0.229, 0.224, 0.225])

# Sanity checks
assert len(mean) == 3 and len(std) == 3, "Invalid image normalization parameters"

# ---- Classification head & threshold ----
image_threshold = float(image_meta.get("threshold", 0.5))
image_head_type = image_meta.get("head_type", "sigmoid")  # expected: sigmoid

# ---- Image preprocessing pipeline ----
image_transform = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std)
])

print("Image branch loaded successfully")
print("Image device:", image_device)
print("Image size:", img_size)
print("Image threshold:", image_threshold)
print("Image head type:", image_head_type)


Image branch loaded successfully
Image device: cpu
Image size: 224
Image threshold: 0.5
Image head type: sigmoid


In [44]:
# =========================================
# IMAGE BRANCH – INFERENCE HELPERS
# =========================================

def read_image_source(source):
    """
    Reads an image from:
    - a filesystem path
    - or (zip_path, inner_path)
    """

    try:
        if isinstance(source, tuple):
            zip_path, inner_path = source
            with zipfile.ZipFile(zip_path, "r") as zf:
                if inner_path not in zf.namelist():
                    raise ValueError(f"Image '{inner_path}' not found in ZIP: {zip_path}")
                img_bytes = zf.read(inner_path)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        else:
            img = Image.open(source).convert("RGB")

        return img

    except Exception as e:
        raise ValueError(
            "Input was routed to image branch but could not be decoded as an image"
        ) from e


def image_branch_predict(sample):
    """
    Inference-only image branch prediction.
    Returns malware probability in [0,1].
    """

    img = read_image_source(sample)
    x = image_transform(img).unsqueeze(0).to(image_device)

    with torch.no_grad():
        logits = image_model(x)
        # Model trained with single-logit sigmoid head
        prob = torch.sigmoid(logits)[0, 0].item()

    return float(prob)


In [45]:
def confidence_weighted_fusion(probs, weights, threshold):
    total_w, score = 0.0, 0.0

    for k, p in probs.items():
        if p is None:
            continue
        w = weights.get(k, 0.0)
        score += w * p
        total_w += w

    if total_w == 0:
        raise ValueError("No valid modality present")

    fused_prob = score / total_w
    fused_pred = int(fused_prob >= threshold)
    return fused_prob, fused_pred


In [51]:
# ===============================
# FUSION PARAMETERS (FROM OPTUNA)
# ===============================
FUSION_WEIGHTS = {
    "image": 0.7929,
    "text":  0.1067,
    "tabular": 0.2171
}
FUSION_THRESHOLD = 0.7499

def unified_fusion_inference(inputs):
    probs = {
        "image": image_branch_predict(inputs["image"]) if inputs.get("image") else None,
        "text": text_branch_predict(inputs["text"]) if inputs.get("text") else None,
        "tabular": tabular_branch_predict(inputs["tabular"]) if inputs.get("tabular") else None,
    }

    return confidence_weighted_fusion(
        probs,
        FUSION_WEIGHTS,
        FUSION_THRESHOLD
    )



**Unified Inference Controller**

In [50]:
def unified_inference(sample):
    features = extract_structural_features(sample)
    X = np.array(features).reshape(1, -1)
    assert X.shape[1] == router.n_features_in_, \
    "Feature dimension mismatch between router training and inference"


    modality = router.predict(X)[0]

    if modality == 0:
        branch = "image"
        prob = image_branch_predict(sample)
        pred = int(prob >= image_threshold)

    elif modality == 1:
        branch = "tabular"
        prob = tabular_branch_predict(sample)
        pred = int(prob >= tab_threshold)

    elif modality == 2:
        branch = "text"
        prob = text_branch_predict(sample)
        pred = int(prob >= text_meta["threshold"])

    return {
        "routed_to": branch,
        "malware_probability": prob,
        "prediction": pred
    }


In [ ]:
# ===============================
# FINAL TESTS
# ===============================
with open("/content/unseen.txt", "w") as f:
    f.write("This malware connects to a remote C2 server")

print("Single-modality test:")
print(unified_inference("/content/unseen.txt"))

print("\nMulti-modality test:")
inputs = {
    "text": "/content/unseen.txt",
    "image": None,
    "tabular": None
}
print(unified_fusion_inference(inputs))
